#06 — Clinical Validation
## Relational multilabel modeling with clinical co-occurrence

**Objective:** Prepare material for validation by an endoscopist.

**Outputs:**
- `project/clinical_validation/validacao_clinica.xlsx` — spreadsheet with embedded images for filling
- `project/clinical_validation/imagens_referencia.pdf` — PDF with larger images
- `project/clinical_validation/amostras/` — individual images organized by case

**Important:** This notebook uses samples from the **test** set (never seen by the model during training/validation). The predictions come from the best model: M2 (Co-occurrence Regularization) - Best Proven Model.

In [ ]:
!pip install openpyxl matplotlib pillow -q

In [ ]:
import json
import shutil
import random
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import (PatternFill, Font, Alignment, Border, Side,
                              GradientFill)
from openpyxl.utils import get_column_letter

# ── Paths ────────────────────────────────── ───────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/workspace')
except:
    ROOT = Path(r"/workspace")

IMGS_DIR    = ROOT / "Dev" / "Data" / "Imgs"
OUT_DIR     = ROOT / 'project'
RESULTS_DIR = OUT_DIR / "results"
SPLITS_DIR  = OUT_DIR / "splits"

VAL_DIR     = OUT_DIR / "clinical_validation"
SAMPLES_DIR = VAL_DIR / "amostras"
VAL_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ────────────────────────────────── ──────────────────────────────────
CORE_COLS = ["ENANTEMA", "PÓLIPO", "ÚLCERA", "EROSÃO", "MICRONODULARIDADE"]
LABEL_PT  = {
    "ENANTEMA":          "Enantema",
    "PÓLIPO":            "Pólipo",
    "ÚLCERA":            "Úlcera",
    "EROSÃO":            "Erosão",
    "MICRONODULARIDADE": "Micronodularidade",
}
LABEL_EN = {
    "ENANTEMA": "Enanthema", "PÓLIPO": "Polyp", "ÚLCERA": "Ulcer",
    "EROSÃO": "Erosion", "MICRONODULARIDADE": "Micronodularity",
}

print("Setup concluído.")
print(f"Saída: {VAL_DIR}")

## 1. Loads predictions from the best model (M2 (Co-occurrence Regularization) - Best Proven Model)

In [ ]:
# ── Loads results from NB05 and recalculates inference from the best model (M2) ──
import json
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path

# Paths needed for inference
ROOT = Path(r"/workspace")
try:
    import os
    if os.path.exists('/workspace'):
        ROOT = Path('/workspace')
except:
    pass

IMGS_DIR    = ROOT / "Dev" / "Data" / "Imgs"
OUT_DIR     = ROOT / 'project'
RESULTS_DIR = OUT_DIR / "results"
SPLITS_DIR  = OUT_DIR / "splits"
CORE_COLS = ["ENANTEMA", "PÓLIPO", "ÚLCERA", "EROSÃO", "MICRONODULARIDADE"]

with open(RESULTS_DIR / "05_relational_results.json", encoding="utf-8") as f:
    r05 = json.load(f)

print("Avaliando o melhor modelo comprovado: M2 (Co-occurrence Regularization)")

# Uses Fold 0 as a base sample for clinical trial extraction
FOLD_REF = 0
# m2_fold = r05["M2_coo"]["per_seed"][str(FOLD_REF)] # Reserved for thresholds if necessary

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_resnet50_m2(n_labels):
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, n_labels))
    return m

class GastroDataset(Dataset):
    def __init__(self, df, imgs_dir, label_cols, transform=None):
        self.df         = df.reset_index(drop=True)
        self.imgs_dir   = Path(imgs_dir)
        self.label_cols = label_cols
        self.transform  = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(self.imgs_dir / row["image_name"]).convert("RGB")
        if self.transform: img = self.transform(img)
        lbl  = torch.tensor(row[self.label_cols].values.astype(float), dtype=torch.float32)
        return img, lbl, row["image_name"]

TF_EVAL = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

df_te = pd.read_csv(SPLITS_DIR / f"fold_{FOLD_REF}_test.csv")
for col in CORE_COLS:
    df_te[col] = df_te[col].fillna(0).astype(int)

loader_te = DataLoader(GastroDataset(df_te, IMGS_DIR, CORE_COLS, TF_EVAL),
                       batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

model = build_resnet50_m2(len(CORE_COLS)).to(DEVICE)
ckpt_path = OUT_DIR / "models" / f"M2_fold{FOLD_REF}.pt"
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

probs_list, tgt_list, nm_list = [], [], []
with torch.no_grad():
    for imgs, lbl, nm in loader_te:
        p = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
        probs_list.append(p); tgt_list.append(lbl.numpy()); nm_list.extend(list(nm))

probs_te = np.vstack(probs_list)
targets_te = np.vstack(tgt_list)

thresholds = ckpt["thresholds"]
preds_te = np.stack([(probs_te[:, i] >= thresholds[c]).astype(int) for i, c in enumerate(CORE_COLS)], axis=1)

img_names = nm_list
probs     = probs_te
preds     = preds_te
targets   = targets_te

print(f"Imagens no conjunto de teste (Fold {FOLD_REF}): {len(img_names)}")
print(f"Thresholds otimizados na validação: {thresholds}")

# Assemble prediction DataFrame
df_preds = pd.DataFrame({
    "image_name": img_names,
    **{f"pred_{c}": preds[:, i].astype(int) for i, c in enumerate(CORE_COLS)},
    **{f"real_{c}": targets[:, i].astype(int) for i, c in enumerate(CORE_COLS)},
    **{f"prob_{c}": probs[:, i].round(3) for i, c in enumerate(CORE_COLS)},
})

# Global hit indicator per image
df_preds["n_acertos"] = sum(
    (df_preds[f"pred_{c}"] == df_preds[f"real_{c}"]).astype(int)
    for c in CORE_COLS
)
df_preds["acerto_total"] = df_preds["n_acertos"] == len(CORE_COLS)

print(f"\nAcerto completo (todas as 5 classes corretas): "
      f"{df_preds['acerto_total'].sum()} / {len(df_preds)} imagens "
      f"({df_preds['acerto_total'].mean()*100:.1f}%)")
display(df_preds.head(3))

## 2. Stratified selection of samples for validation

In [ ]:
# ── Stratified selection: cover the most informative cases ──────────────────
# Selection criteria (all exclusive to the test, not seen in training):
#   A) True Positives by class (model correct present pathology)
#   B) False Negatives by class (model made a mistake — missed the pathology)
#   C) False Positives by class (model signaled no real pathology)
#   D) Images with multiple pathologies (real multilabel ≥ 2)
#   E) Correctly predicted normal images (all negative)

random.seed(42)
np.random.seed(42)

N_PER_CATEGORY = 3   # samples by category by class
N_NORMAL       = 5   # normal/negative images

selected_indices = set()
sample_meta = []   # metadata of each selected sample

def pick(mask_series, n, label=None, category=None):
    candidates = mask_series[mask_series].index.tolist()
    chosen = random.sample(candidates, min(n, len(candidates)))
    for idx in chosen:
        if idx not in selected_indices:
            selected_indices.add(idx)
            sample_meta.append({
                "index": idx,
                "categoria": category,
                "patologia": label,
            })
    return chosen

for col in CORE_COLS:
    tp_mask = (df_preds[f"pred_{col}"] == 1) & (df_preds[f"real_{col}"] == 1)
    fn_mask = (df_preds[f"pred_{col}"] == 0) & (df_preds[f"real_{col}"] == 1)
    fp_mask = (df_preds[f"pred_{col}"] == 1) & (df_preds[f"real_{col}"] == 0)
    pick(tp_mask, N_PER_CATEGORY, label=col, category="TP")
    pick(fn_mask, N_PER_CATEGORY, label=col, category="FN")
    pick(fp_mask, N_PER_CATEGORY, label=col, category="FP")

# Multilabel (≥2 real pathologies)
ml_mask = pd.Series(
    targets.sum(axis=1) >= 2, index=df_preds.index)
pick(ml_mask, 5, label="MULTILABEL", category="MULTILABEL")

# Normals (all 5 = 0)
neg_mask = pd.Series(
    targets.sum(axis=1) == 0, index=df_preds.index)
pick(neg_mask, N_NORMAL, label="NORMAL", category="NORMAL")

# ── Edge cases (flagged_extreme_images.csv_not_used)
extreme_path = SPLITS_DIR / "flagged_extreme_images.csv_not_used"
if extreme_path.exists():
    df_ext = pd.read_csv(extreme_path)
    # Select only those in the test set (df_preds)
    ext_test = df_preds[df_preds["image_name"].isin(df_ext["image_name"])]
    for idx in ext_test.index:
        if idx not in selected_indices:
            selected_indices.add(idx)
            sample_meta.append({
                "index": idx,
                "categoria": "EXTREMO",
                "patologia": "MULTIPLA"
            })

# Assembles final DataFrame from samples (without repetition)
df_samples = df_preds.loc[sorted(selected_indices)].copy().reset_index(drop=True)
df_samples["caso_id"] = [f"CASO-{i+1:03d}" for i in range(len(df_samples))]

print(f"Total de casos selecionados: {len(df_samples)}")
print(f"  TP/FN/FP por classe: até {N_PER_CATEGORY} cada")
print(f"  Multilabel: até 5")
print(f"  Normais: até {N_NORMAL}")

# Category distribution
for cat in ["TP","FN","FP","MULTILABEL","NORMAL", "EXTREMO"]:
    n = sum(1 for m in sample_meta if m["categoria"] == cat and m["index"] in selected_indices)
    print(f"  {cat}: {n} casos")

In [ ]:
# ── Copy images to samples folder/ with standardized name ──────────────────
for _, row in df_samples.iterrows():
    src = IMGS_DIR / row["image_name"]
    dst = SAMPLES_DIR / f"{row['caso_id']}_{row['image_name']}"
    if src.exists():
        shutil.copy2(src, dst)

print(f"Imagens copiadas para: {SAMPLES_DIR}")
print(f"Total: {len(list(SAMPLES_DIR.iterdir()))} arquivos")

## 3. Excel spreadsheet for the endoscopist

In [ ]:
# ── Style Helpers ──────────────────────────── ─────────────────────────────
def _border(style="thin"):
    s = Side(style=style, color="BFBFBF")
    return Border(left=s, right=s, top=s, bottom=s)

def _fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

def _font(bold=False, size=10, color="000000"):
    return Font(bold=bold, size=size, color=color, name="Calibri")

def _align(h="center", v="center", wrap=True):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap)

# Palette
C_HEADER_DARK  = "1F3864"   # dark blue (main header)
C_HEADER_MED   = "2E75B6"   # medium blue (sub-header)
C_HEADER_LIGHT = "D6E4F0"   # light blue (even line)
C_GREEN_HEAD   = "375623"   # dark green (validation section)
C_GREEN_LIGHT  = "E2EFDA"   # light green (fill cells)
C_YELLOW       = "FFF2CC"   # yellow (confidence)
C_RED_LIGHT    = "FCE4D6"   # light red (discordance)
C_WHITE        = "FFFFFF"
C_GRAY_LIGHT   = "F2F2F2"

def sim_nao_options(ws, cell_ref):
    """Adiciona validação de lista Sim/Não a uma célula."""
    from openpyxl.worksheet.datavalidation import DataValidation
    dv = DataValidation(type="list", formula1='"Sim,Não,Parcial"',
                        allow_blank=True, showDropDown=False)
    ws.add_data_validation(dv)
    dv.add(cell_ref)

print("Helpers de estilo definidos.")

In [ ]:
# ── Builds Excel ───────────────────────────── ─────────────────────────────
# Structure: each case occupies 1 line height 120 px (image) + 2 lines of data
# Columns:
#   A = Case number
#   B = Image (embedded)
#   C–G = What the model detected (YES/NO by pathology)
#   H–L = Model probability (0–100%)
#   M = General agreement (Yes/No/Partial dropdown) ← clinician fills in
#   N–R = Agreement by pathology (dropdown) ← clinician fills in
#   S = Free observations ← clinician fills in
#   T = Answer key (original annotation) — hidden by default

wb = openpyxl.Workbook()

# ── Tab 1: Instructions ──────────────────────────── ─────────────────────────────
ws_inst = wb.active
ws_inst.title = "INSTRUÇÕES"
ws_inst.sheet_view.showGridLines = False
ws_inst.column_dimensions["A"].width = 2
ws_inst.column_dimensions["B"].width = 90

instr_lines = [
    ("VALIDAÇÃO CLÍNICA — SISTEMA DE DETECÇÃO DE ACHADOS ENDOSCÓPICOS", True, 16, C_HEADER_DARK),
    ("", False, 10, C_WHITE),
    ("OBJETIVO", True, 12, C_HEADER_MED),
    ("Este material apresenta imagens de gastroscopia com as detecções automáticas realizadas pelo modelo de inteligência artificial.", False, 10, "000000"),
    ("Solicitamos sua avaliação sobre a concordância clínica das predições.", False, 10, "000000"),
    ("", False, 10, C_WHITE),
    ("COMO PREENCHER", True, 12, C_HEADER_MED),
    ("1. Acesse a aba  VALIDAÇÃO  (próxima aba)", False, 10, "000000"),
    ("2. Para cada caso, visualize a imagem e as detecções do modelo (colunas em azul)", False, 10, "000000"),
    ("3. Preencha apenas as colunas em VERDE:", False, 10, "000000"),
    ("   • Concordância Geral: o modelo acertou de modo geral? (Sim / Não / Parcial)", False, 10, "000000"),
    ("   • Concordância por Achado: para cada patologia, o modelo acertou? (Sim / Não)", False, 10, "000000"),
    ("   • Observações: escreva livremente qualquer comentário clínico relevante", False, 10, "000000"),
    ("", False, 10, C_WHITE),
    ("LEGENDA DAS DETECÇÕES", True, 12, C_HEADER_MED),
    ("   SIM  = modelo detectou o achado nesta imagem", False, 10, "000000"),
    ("   NÃO  = modelo não detectou o achado nesta imagem", False, 10, "000000"),
    ("   %    = confiança do modelo (0% = incerto, 100% = muito confiante)", False, 10, "000000"),
    ("", False, 10, C_WHITE),
    ("ACHADOS AVALIADOS", True, 12, C_HEADER_MED),
    ("   • Enantema — hiperemia/eritema da mucosa", False, 10, "000000"),
    ("   • Pólipo — lesão elevada da mucosa", False, 10, "000000"),
    ("   • Úlcera — solução de continuidade da mucosa", False, 10, "000000"),
    ("   • Erosão — lesão superficial da mucosa sem atingir muscular", False, 10, "000000"),
    ("   • Micronodularidade — padrão nodular fino da mucosa", False, 10, "000000"),
    ("", False, 10, C_WHITE),
    ("CONFIDENCIALIDADE", True, 12, C_HEADER_MED),
    ("As imagens são provenientes de banco de dados anonimizado. Uso exclusivamente científico.", False, 10, "000000"),
]

for row_i, (text, bold, size, color) in enumerate(instr_lines, start=2):
    cell = ws_inst.cell(row=row_i, column=2, value=text)
    cell.font = Font(bold=bold, size=size, color=color, name="Calibri")
    cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    ws_inst.row_dimensions[row_i].height = 20 if text else 8

ws_inst.row_dimensions[2].height = 30

print("Aba de instruções criada.")

In [ ]:
# ── Tab 2: Validation ───────────────────────────── ─────────────────────────────
ws = wb.create_sheet("VALIDAÇÃO")
ws.sheet_view.showGridLines = False
ws.freeze_panes = "C4"   # freezes headers

# Column widths
col_widths = {
    "A": 10,   # Case
    "B": 22,   # Image
    "C": 14, "D": 14, "E": 14, "F": 14, "G": 14,   # YES/NO detection
    "H": 10, "I": 10, "J": 10, "K": 10, "L": 10,   # Probability %
    "M": 18,   # General Agreement (fill in)
    "N": 14, "O": 14, "P": 14, "Q": 14, "R": 14,   # Agreement by finding
    "S": 40,   # Observations
    "T": 20,   # Template (hidden)
}
for col_letter, width in col_widths.items():
    ws.column_dimensions[col_letter].width = width

# ── Header Line 1: groups ──────────────────────── ─────────────────────────
grupos = [
    (1, 1, ""),
    (2, 2, ""),
    (3, 7,  "DETECÇÃO DO MODELO"),
    (8, 12, "CONFIANÇA DO MODELO (%)"),
    (13, 13,"← PREENCHA"),
    (14, 18,"← PREENCHA — CONCORDÂNCIA POR ACHADO"),
    (19, 19,"← PREENCHA — OBSERVAÇÕES"),
    (20, 20,"GABARITO"),
]
for c1, c2, label in grupos:
    ws.merge_cells(start_row=1, start_column=c1, end_row=1, end_column=c2)
    cell = ws.cell(row=1, column=c1, value=label)
    if "PREENCHA" in label:
        cell.fill    = _fill(C_GREEN_HEAD)
        cell.font    = _font(bold=True, size=10, color="FFFFFF")
    elif label:
        cell.fill    = _fill(C_HEADER_DARK)
        cell.font    = _font(bold=True, size=10, color="FFFFFF")
    cell.alignment = _align()
    cell.border    = _border()

# ── Header Line 2: sub-labels ────────────────────── ───────────────────────
label_names_pt = [LABEL_PT[c] for c in CORE_COLS]
headers_row2 = (
    ["Caso", "Imagem endoscópica"]
    + [f"{n}\nDetectado?" for n in label_names_pt]
    + [f"{n}\nConfiança" for n in label_names_pt]
    + ["Concordância\nGeral"]
    + [f"{n}\nConcordância" for n in label_names_pt]
    + ["Observações clínicas\n(texto livre)"]
    + ["Anotação\nOriginal"]
)
for col_i, label in enumerate(headers_row2, start=1):
    cell = ws.cell(row=2, column=col_i, value=label)
    if col_i == 13:
        cell.fill = _fill(C_GREEN_HEAD)
        cell.font = _font(bold=True, size=9, color="FFFFFF")
    elif col_i in range(14, 20):
        cell.fill = _fill(C_GREEN_HEAD)
        cell.font = _font(bold=True, size=9, color="FFFFFF")
    elif col_i == 19:
        cell.fill = _fill(C_GREEN_HEAD)
        cell.font = _font(bold=True, size=9, color="FFFFFF")
    elif col_i in range(3, 8):
        cell.fill = _fill(C_HEADER_MED)
        cell.font = _font(bold=True, size=9, color="FFFFFF")
    elif col_i in range(8, 13):
        cell.fill = _fill("4472C4")
        cell.font = _font(bold=True, size=9, color="FFFFFF")
    else:
        cell.fill = _fill(C_HEADER_DARK)
        cell.font = _font(bold=True, size=9, color="FFFFFF")
    cell.alignment = _align()
    cell.border    = _border()

ws.row_dimensions[1].height = 22
ws.row_dimensions[2].height = 36

print("Cabeçalhos criados.")

In [ ]:
# ── Fill in case lines ──────────────────────── ─────────────────────────
IMG_ROW_HEIGHT = 115  # height pixels of line with image
IMG_SIZE_PX    = 130  # inline thumbnail size

from openpyxl.worksheet.datavalidation import DataValidation
dv_geral = DataValidation(type="list", formula1='"Sim,Não,Parcial"',
                          allow_blank=True, showDropDown=False)
dv_achado = DataValidation(type="list", formula1='"Sim,Não"',
                           allow_blank=True, showDropDown=False)
ws.add_data_validation(dv_geral)
ws.add_data_validation(dv_achado)

for row_num, (_, sample) in enumerate(df_samples.iterrows(), start=3):
    excel_row = row_num
    ws.row_dimensions[excel_row].height = IMG_ROW_HEIGHT
    is_even = (row_num % 2 == 0)
    bg = C_HEADER_LIGHT if is_even else C_WHITE

    # Col A: Case ID
    c = ws.cell(row=excel_row, column=1, value=sample["caso_id"])
    c.font = _font(bold=True, size=10)
    c.fill = _fill(C_HEADER_DARK)
    c.font = _font(bold=True, size=10, color="FFFFFF")
    c.alignment = _align()
    c.border = _border()

    # Col B: Embedded image
    img_path = IMGS_DIR / sample["image_name"]
    if img_path.exists():
        try:
            pil_img = Image.open(img_path).convert("RGB")
            # Resize maintaining proportion
            pil_img.thumbnail((IMG_SIZE_PX, IMG_SIZE_PX), Image.LANCZOS)
            buf = io.BytesIO()
            pil_img.save(buf, format="PNG")
            buf.seek(0)
            xl_img = XLImage(buf)
            xl_img.anchor = f"B{excel_row}"
            ws.add_image(xl_img)
        except Exception as e:
            ws.cell(row=excel_row, column=2, value=f"[{sample['image_name']}]")
    ws.cell(row=excel_row, column=2).border = _border()

    # Cols C–G: YES/NO Detection
    for ci, col in enumerate(CORE_COLS, start=3):
        val = sample[f"pred_{col}"]
        text = "✔ SIM" if val == 1 else "✘ NÃO"
        c = ws.cell(row=excel_row, column=ci, value=text)
        c.fill = _fill("C6EFCE" if val == 1 else "FFCCCC")
        c.font = _font(bold=True, size=10,
                       color="375623" if val == 1 else "9C0006")
        c.alignment = _align()
        c.border = _border()

    # Cols H–L: Probability (%)
    for ci, col in enumerate(CORE_COLS, start=8):
        prob_val = sample[f"prob_{col}"]
        pct = f"{prob_val*100:.0f}%"
        c = ws.cell(row=excel_row, column=ci, value=pct)
        # Color gradient: green > 70%, yellow 40-70%, red < 40%
        p = prob_val
        if p >= 0.70:   hex_c = "C6EFCE"
        elif p >= 0.40: hex_c = "FFEB9C"
        else:           hex_c = "FFCCCC"
        c.fill = _fill(hex_c)
        c.font = _font(size=10)
        c.alignment = _align()
        c.border = _border()

    # Col M: General Agreement (dropdown — fill in)
    c = ws.cell(row=excel_row, column=13, value="")
    c.fill = _fill(C_GREEN_LIGHT)
    c.font = _font(size=10)
    c.alignment = _align()
    c.border = _border("medium")
    dv_geral.add(f"M{excel_row}")

    # Cols N–R: Agreement by finding (dropdown — fill in)
    for ci in range(14, 19):
        c = ws.cell(row=excel_row, column=ci, value="")
        c.fill = _fill(C_GREEN_LIGHT)
        c.font = _font(size=10)
        c.alignment = _align()
        c.border = _border("medium")
        dv_achado.add(f"{get_column_letter(ci)}{excel_row}")

    # Col S: Free observations
    c = ws.cell(row=excel_row, column=19, value="")
    c.fill = _fill(C_GREEN_LIGHT)
    c.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
    c.border = _border("medium")

    # Col T: Template (original annotation)
    real_labels = [LABEL_PT[col] for col in CORE_COLS
                   if sample[f"real_{col}"] == 1]
    gabarito = ", ".join(real_labels) if real_labels else "Sem achados"
    c = ws.cell(row=excel_row, column=20, value=gabarito)
    c.fill = _fill(C_GRAY_LIGHT)
    c.font = _font(size=9, color="595959")
    c.alignment = _align(h="left")
    c.border = _border()

print(f"Linhas preenchidas: {len(df_samples)} casos")

# Hides answer key column by default (reviewer fills in without seeing first)
ws.column_dimensions["T"].hidden = True

In [ ]:
# ── Tab 3: Summary (for the clinician to see progress) ───────────────────────────
ws_res = wb.create_sheet("RESUMO")
ws_res.sheet_view.showGridLines = False
ws_res.column_dimensions["A"].width = 30
ws_res.column_dimensions["B"].width = 20
ws_res.column_dimensions["C"].width = 20

resumo_header = ["Informação", "Valor", "Detalhes"]
for ci, h in enumerate(resumo_header, start=1):
    c = ws_res.cell(row=1, column=ci, value=h)
    c.fill = _fill(C_HEADER_DARK)
    c.font = _font(bold=True, color="FFFFFF")
    c.alignment = _align()
    c.border = _border()

resumo_data = [
    ("Total de casos para revisar", str(len(df_samples)), ""),
    ("Modelo utilizado", "M2 (ResNet50 + L_coo)", "Co-occurrence Regularization (melhor modelo comprobatório)"),
    ("Achados avaliados", "5", "Enantema, Pólipo, Úlcera, Erosão, Micronodularidade"),
    ("Fonte das imagens", "Conjunto de teste", "Nunca vistas pelo modelo no treino"),
    ("", "", ""),
    ("INSTRUÇÕES DE PREENCHIMENTO", "", ""),
    ("1. Abra a aba VALIDAÇÃO", "", ""),
    ("2. Para cada caso: veja a imagem + detecções", "", ""),
    ("3. Clique nas células VERDES e selecione Sim/Não", "", ""),
    ("4. Escreva observações na coluna de texto livre", "", ""),
    ("5. Salve o arquivo ao final", "", ""),
]

for ri, (a, b, c_val) in enumerate(resumo_data, start=2):
    ws_res.cell(row=ri, column=1, value=a).font = _font(bold=(a.startswith("INSTRUC") or a.startswith("Total")))
    ws_res.cell(row=ri, column=2, value=b)
    ws_res.cell(row=ri, column=3, value=c_val)
    for ci in range(1, 4):
        ws_res.cell(row=ri, column=ci).alignment = _align(h="left")
        ws_res.cell(row=ri, column=ci).border = _border()

# Save Excel
xl_path = VAL_DIR / "validacao_clinica.xlsx"
wb.save(xl_path)
print(f"✓ Excel salvo: {xl_path}")
print(f"  Tamanho: {xl_path.stat().st_size / 1024:.0f} KB")

## 4. Reference PDF — full size images

In [ ]:
# ── PDF with 4 cases per page, large image + detection table ───────────
CASES_PER_PAGE = 4
pdf_path = VAL_DIR / "imagens_referencia.pdf"

with PdfPages(pdf_path) as pdf:
    n_pages = (len(df_samples) + CASES_PER_PAGE - 1) // CASES_PER_PAGE

    for page in range(n_pages):
        chunk = df_samples.iloc[page*CASES_PER_PAGE : (page+1)*CASES_PER_PAGE]

        fig = plt.figure(figsize=(11.7, 16.5))   # A4
        fig.patch.set_facecolor("#F8F9FA")

        # Page title
        fig.text(0.5, 0.975,
                 "VALIDAÇÃO CLÍNICA — SISTEMA DE DETECÇÃO DE ACHADOS ENDOSCÓPICOS",
                 ha="center", va="top", fontsize=12, fontweight="bold", color="#1F3864")
        fig.text(0.5, 0.960,
                 f"Página {page+1} de {n_pages}  |  Modelo: M2 (Co-occurrence Regularization)  |  Conjunto: Teste",
                 ha="center", va="top", fontsize=8, color="#595959")

        outer_gs = gridspec.GridSpec(CASES_PER_PAGE, 1,
                                     figure=fig,
                                     top=0.945, bottom=0.02,
                                     hspace=0.08)

        for local_i, (_, row) in enumerate(chunk.iterrows()):
            inner_gs = gridspec.GridSpecFromSubplotSpec(
                1, 2, subplot_spec=outer_gs[local_i],
                width_ratios=[1, 2.2], wspace=0.04)

            # Image
            ax_img = fig.add_subplot(inner_gs[0])
            img_path = IMGS_DIR / row["image_name"]
            if img_path.exists():
                try:
                    im = Image.open(img_path).convert("RGB")
                    ax_img.imshow(np.array(im))
                except:
                    ax_img.text(0.5, 0.5, "[sem imagem]", ha="center")
            ax_img.set_title(f"{row['caso_id']}",
                             fontsize=9, fontweight="bold", color="#1F3864", pad=4)
            ax_img.axis("off")
            for spine in ax_img.spines.values():
                spine.set_edgecolor("#BFBFBF"); spine.set_linewidth(0.8)
                spine.set_visible(True)

            # Detection table
            ax_tab = fig.add_subplot(inner_gs[1])
            ax_tab.axis("off")

            col_labels = ["Achado", "Detectado?", "Confiança"]
            tab_data = []
            for c_label in CORE_COLS:
                detected = "✔ SIM" if row[f"pred_{c_label}"] == 1 else "✘ NÃO"
                prob_pct  = f"{row[f'prob_{c_label}']*100:.0f}%"
                tab_data.append([LABEL_PT[c_label], detected, prob_pct])

            table = ax_tab.table(
                cellText=tab_data,
                colLabels=col_labels,
                cellLoc="center",
                loc="center",
                bbox=[0, 0, 1, 1]
            )
            table.auto_set_font_size(False)
            table.set_fontsize(8.5)

            # Table style
            for (r, c), cell in table.get_celld().items():
                cell.set_edgecolor("#BFBFBF")
                if r == 0:  # header
                    cell.set_facecolor("#1F3864")
                    cell.set_text_props(color="white", fontweight="bold")
                else:
                    if c == 1:  # Detected column
                        text = tab_data[r-1][1]
                        cell.set_facecolor("#C6EFCE" if "SIM" in text else "#FFCCCC")
                    elif c == 2:  # trust
                        p_val = float(tab_data[r-1][2].replace("%","")) / 100
                        if p_val >= 0.70:   cell.set_facecolor("#C6EFCE")
                        elif p_val >= 0.40: cell.set_facecolor("#FFEB9C")
                        else:               cell.set_facecolor("#FFCCCC")
                    else:
                        cell.set_facecolor("#F2F2F2" if r%2==0 else "white")

            # Space for handwritten observation
            ax_tab.text(0.0, -0.08,
                        "Observação: _______________________________________________",
                        transform=ax_tab.transAxes,
                        fontsize=7.5, color="#595959", va="top")
            ax_tab.text(0.0, -0.16,
                        "Concordância geral:  [ ] Sim   [ ] Não   [ ] Parcial",
                        transform=ax_tab.transAxes,
                        fontsize=7.5, color="#595959", va="top")

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"✓ PDF salvo: {pdf_path}")
print(f"  Páginas: {n_pages}  |  Tamanho: {pdf_path.stat().st_size / 1024:.0f} KB")

## 5. Summary and delivery checklist

In [ ]:
print("=" * 60)
print("NB06 — VALIDAÇÃO CLÍNICA — RESUMO DE ENTREGA")
print("=" * 60)
print(f"\nCasos selecionados: {len(df_samples)}")
print(f"  Fontes: TP / FN / FP por classe + multilabel + normais")
print(f"  Conjunto: TESTE (imagens nunca vistas no treino)")
print(f"  Modelo: M2 (Co-occurrence Regularization) (Fold 0)")
print()
print("Arquivos gerados:")
print(f"  ✓ {VAL_DIR/'validacao_clinica.xlsx'}")
print(f"      → Abrir no Excel/Google Sheets")
print(f"      → 3 abas: INSTRUÇÕES / VALIDAÇÃO / RESUMO")
print(f"      → Imagens embutidas, células verdes para preencher")
print(f"      → Dropdowns Sim/Não/Parcial nas colunas de concordância")
print()
print(f"  ✓ {VAL_DIR/'imagens_referencia.pdf'}")
print(f"      → PDF A4, 4 casos por página")
print(f"      → Imagem grande + tabela de detecções colorida")
print(f"      → Espaço para anotação manuscrita")
print()
print(f"  ✓ {SAMPLES_DIR}/")
print(f"      → {len(list(SAMPLES_DIR.iterdir()))} imagens individuais (CASO-XXX_nome.jpg)")
print()
print("O que o clínico precisa receber:")
print("  • validacao_clinica.xlsx  (arquivo principal)")
print("  • imagens_referencia.pdf  (caso prefira impresso)")
print("  • [opcional] pasta amostras/ para ver imagens maiores")
print()
print("Após receber de volta:")
print("  → Carregar validacao_clinica.xlsx preenchido")
print("  → Calcular Cohen's Kappa e concordância por achado")
print("  → Incluir no artigo como validação externa")
print("=" * 60)